## E3 — Latent space visualization (μ) + kNN purity (Kaggle)\n\nThis notebook extracts encoder mean vectors **μ** for `z_p` from the **test memmap split**, then runs:\n- PCA → UMAP (2D)\n- colors by attribute bins (mean-over-bars)\n- computes a simple quantitative score: **kNN purity** in latent space\n\nOutputs are written to `artifacts/eval_latent_viz/`:\n- `latent_embeddings.npz` (μ + reduced attributes)\n- `umap_by_attr_*.png`\n- `latent_knn_purity.json`\n

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import numpy as np
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

IS_KAGGLE = Path('/kaggle').exists() and Path('/kaggle/input').exists()
WORKING_ROOT = Path('/kaggle/working') if IS_KAGGLE else (Path.cwd() / 'artifacts' / 'local_eval')
OUT_DIR = (WORKING_ROOT / 'artifacts' / 'eval_latent_viz').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR:', OUT_DIR)

# Repo imports
import sys
repo_root = (Path.cwd().parents[1] if (Path.cwd().name == 'notebooks') else Path.cwd())
ml_src = (repo_root / 'ml' / 'src').resolve()
if str(ml_src) not in sys.path:
    sys.path.insert(0, str(ml_src))

from musicgen.analysis.latent_space_viz import LatentVizConfig, run_latent_space_viz


In [ ]:
# Use your best checkpoints here.
cfg = LatentVizConfig(
    n_samples=1500,
    batch_size=64,
    seed=123,
    ckpt_vae_v3='ml/src/musicgen/runs/vae_v3/ckpt.pt',
    ckpt_simple_v2='ml/src/musicgen/runs/simple_v2/ckpt.pt',
    out_dir=str(OUT_DIR),
    embedding='mu',
    pca_dim=50,
    umap_neighbors=30,
    umap_min_dist=0.05,
)

report_path = run_latent_space_viz(cfg)
report_path


In [ ]:
report = json.loads(Path(report_path).read_text(encoding='utf-8'))
list(report.keys())


In [ ]:
# kNN purity: do nearest neighbors share the same label?
# We compute purity for the 4 mean-over-bars attribute bins (attrs_mean[:, j]).

from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

emb_path = OUT_DIR / 'latent_space_embeddings.npz'
assert emb_path.exists(), f'Missing {emb_path} (run the extraction cell above)'
z = np.load(emb_path)

attrs_mean = z['attrs_mean'].astype(np.int64)  # [N,4]
vae_mu = z['vae_mu'].astype(np.float32)        # [N,128]
simple_mu = z['simple_mu'].astype(np.float32)  # [N,128]

def knn_purity(emb: np.ndarray, labels: np.ndarray, k: int = 15) -> float:
    X = StandardScaler().fit_transform(emb)
    X = PCA(n_components=min(50, X.shape[1]), random_state=123).fit_transform(X)

    nn = NearestNeighbors(n_neighbors=k + 1, metric='euclidean')
    nn.fit(X)
    idx = nn.kneighbors(X, return_distance=False)[:, 1:]

    matches = (labels[idx] == labels[:, None])
    return float(matches.mean())


knn_report = {
    'k': 15,
    'vae_v3': {},
    'simple_v2': {},
}

attr_names = ['polyphony', 'rhythmic_intensity', 'velocity', 'note_density']
for j, nm in enumerate(attr_names):
    knn_report['vae_v3'][nm] = knn_purity(vae_mu, attrs_mean[:, j], k=15)
    knn_report['simple_v2'][nm] = knn_purity(simple_mu, attrs_mean[:, j], k=15)

out_path = OUT_DIR / 'latent_knn_purity.json'
out_path.write_text(json.dumps(knn_report, indent=2), encoding='utf-8')
knn_report
